# Step 7: Compare the two Power Savings plan PDFs

Select **.venv**, restart the kernel after code changes, and run from the top.
We use your downloaded PDFs, saved GPT-5 extractions and Notebook 04's synthetic
usage. **No new API calls.** No manual JSON completion is required for this lesson.

This is a hypothetical recurring-cost comparison, not a verified current offer or
complete bill. The downloaded PDFs are the source of truth: web links can change
contents while keeping their filenames. We display their actual dates below.

## 1. Imports and input files
The new adapter reads this specific Power Savings layout. It supports an inclusive
monthly usage-credit threshold and preserves evidence for every parsed field.
It rejects missing or conflicting values rather than supplying defaults.
The reusable monthly arithmetic is the same function used in Notebook 04.

In [2]:
from pathlib import Path
from decimal import Decimal
from datetime import datetime, timezone
import json
from html import escape
from IPython.display import display, HTML
from electricity_optimizer.ingestion import read_pdf
from electricity_optimizer.agreements import ExtractionRecord
from electricity_optimizer.extraction import validate_terms
from electricity_optimizer.efl_comparison import parse_power_savings, compare_efls
from electricity_optimizer.pricing import calculate_month
from electricity_optimizer.usage import load_usage_csv, MonthlyUsage, UsageYear

PROJECT_ROOT = Path.cwd()
filenames = ["R1F00166003766B.pdf", "R1F00160000021A.pdf"]
usage_path = PROJECT_ROOT / "output" / "lesson04_synthetic" / "SYNTHETIC_usage_2025.csv"
if not usage_path.exists():
    raise FileNotFoundError("Run Notebook 04 to create the synthetic usage CSV first.")

def table(headers, rows):
    def row(values, tag):
        return "<tr>" + "".join(f"<{tag} style='padding:6px 10px;text-align:left'>{escape(str(v))}</{tag}>" for v in values) + "</tr>"
    display(HTML("<table>" + row(headers, "th") + "".join(row(r, "td") for r in rows) + "</table>"))

## 2. Load PDFs and check saved extraction citations
Match records by filename **and file fingerprint**. If multiple valid records exist,
choose the latest saved extraction timestamp and display the selected file. Source
mismatches are rejected. All model citation issues remain visible.

The numerical demonstration below uses a separate narrow source parser, not the
model's free-text interpretations. A flagged model citation is not silently repaired
or marked approved. This lets us study AI extraction quality alongside exact labeled
price extraction.

In [3]:
documents = []
records = []
audit = []
for filename in filenames:
    document = read_pdf(PROJECT_ROOT / "contracts" / filename, include_tables=True)
    matches = []
    for path in (PROJECT_ROOT / "output" / "extraction").glob(f"{Path(filename).stem}_*.json"):
        candidate = ExtractionRecord.model_validate_json(path.read_text(encoding="utf-8"))
        if candidate.source_sha256 == document.sha256 and candidate.source_file == filename:
            matches.append((datetime.fromisoformat(candidate.extracted_at), path, candidate))
    if not matches:
        raise ValueError(f"No saved extraction matches {filename}. Extract and save it in Notebook 02 first.")
    _, path, record = max(matches, key=lambda item: (item[0], item[1].name))
    issues = validate_terms(record.terms, document)
    print("\nSelected:", path.name)
    for issue in issues:
        print(issue.severity, issue.field, issue.message)
    documents.append(document)
    records.append(record)
    audit.append({"source_file": filename, "source_sha256": document.sha256,
        "extraction_file": str(path), "response_id": record.response_id,
        "issues": [issue.model_dump() for issue in issues]})

Consider using the pymupdf_layout package for a greatly improved page layout analysis.

Selected: R1F00166003766B_resp_0524d91f76fa8efe016aa47cd1fbf487d191fd23340daa0b89.json
warning customer_type Not stated: obtain supporting documents before relying on this term.
error credits_and_minimum_usage Quote not found on page 1.
warning renewal_terms Not stated: obtain supporting documents before relying on this term.

Selected: R1F00160000021A_resp_013d02c2f25ef043016aa47d79cf8487d185d1da3a719914e7.json
warning customer_type Not stated: obtain supporting documents before relying on this term.
error credits_and_minimum_usage Quote not found on page 1.
warning renewal_terms Not stated: obtain supporting documents before relying on this term.


## 3. Inspect source-backed pricing candidates
All numbers come from the loaded PDFs. The adapter converts cents to dollars and
separates the provider's base fee from delivery charges. Currency is interpreted
as USD from the Texas context; it is not an explicit ISO currency declaration.

The scope assumes the standard AEP Central service area, not the special
McAllen/Mission former-Oncor exception. We use listed recurring delivery totals
without estimating separate ITR refunds or tariff changes. See all conditions in
the original PDFs. These are demonstration candidates, not human-approved plans.

In [4]:
plans = [parse_power_savings(document) for document in documents]
table(["Plan", "PDF date", "Territory", "Energy $/kWh", "Base $", "Delivery $/kWh", "Delivery $/cycle", "Credit $", "Threshold kWh"],
      [(p.name, p.document_date, p.market, p.energy_usd_per_kwh, p.base_usd_per_month,
        p.delivery_usd_per_kwh, p.delivery_usd_per_month, p.credit_usd, p.credit_min_kwh) for p in plans])
for plan in plans:
    print("\nEvidence for:", plan.name)
    table(["Field", "Normalized value", "Page", "Exact supporting text"],
          [(field, evidence.value, evidence.page_number, evidence.quote)
           for field, evidence in plan.evidence.items()])

Plan,PDF date,Territory,Energy $/kWh,Base $,Delivery $/kWh,Delivery $/cycle,Credit $,Threshold kWh
"Reliant Power Savings 2,000 kWh 12 plan",09/01/2026,AEP Texas Central,0.166483,0.00,0.057554,3.24,150.00,2000
"Reliant Power Savings 2,000 kWh 24 plan",09/01/2026,AEP Texas Central,0.154124,0.00,0.057554,3.24,150.00,2000



Evidence for: Reliant Power Savings 2,000 kWh 12 plan


Field,Normalized value,Page,Exact supporting text
name,"Reliant Power Savings 2,000 kWh 12 plan",1,"Reliant Power Savings 2,000 kWh 12 plan"
market,AEP Texas Central,1,AEP Texas Central service area
document_date,09/01/2026,1,Date: 09/01/2026
energy_usd_per_kwh,0.166483,1,Energy Charge: 16.6483¢ per kWh
base_usd_per_month,0.00,1,Base Charge: $0.00 per billing cycle
delivery_usd_per_month,3.24,1,AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh
delivery_usd_per_kwh,0.057554,1,AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh
credit_usd,150.00,1,"A Usage Credit of $150.00 will be included for each billing cycle when your usage on this plan is above or equal to 2,000 kWh"
credit_min_kwh,2000,1,"A Usage Credit of $150.00 will be included for each billing cycle when your usage on this plan is above or equal to 2,000 kWh"
contract_months,12,1,Contract Term 12 months



Evidence for: Reliant Power Savings 2,000 kWh 24 plan


Field,Normalized value,Page,Exact supporting text
name,"Reliant Power Savings 2,000 kWh 24 plan",1,"Reliant Power Savings 2,000 kWh 24 plan"
market,AEP Texas Central,1,AEP Texas Central service area
document_date,09/01/2026,1,Date: 09/01/2026
energy_usd_per_kwh,0.154124,1,Energy Charge: 15.4124¢ per kWh
base_usd_per_month,0.00,1,Base Charge: $0.00 per billing cycle
delivery_usd_per_month,3.24,1,AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh
delivery_usd_per_kwh,0.057554,1,AEP Texas Central Delivery Charges: $3.24 per billing cycle and 5.7554¢ per kWh
credit_usd,150.00,1,"A Usage Credit of $150.00 will be included for each billing cycle when your usage on this plan is above or equal to 2,000 kWh"
credit_min_kwh,2000,1,"A Usage Credit of $150.00 will be included for each billing cycle when your usage on this plan is above or equal to 2,000 kWh"
contract_months,24,1,Contract Term 24 months


## 4. Run the comparison through LangGraph
We connect three local steps: check plan compatibility, calculate, and explain.
Compatibility here means same territory, currency and document date. We compare
only the first 12 months, not the total cost of unequal contract lengths.

Assumptions: one usage row per billing cycle, constant rates from the PDF snapshot,
no taxes/deposits/late fees, and no switching charge in the base case. Each component
rounds to cents half up. The inherited engine caps credits at subtotal; with these
plans the qualifying subtotal exceeds $150, so the cap does not change results.
The synthetic usage year is a consumption pattern, not a historical 2025 bill forecast.

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class ComparisonState(TypedDict, total=False):
    usage: UsageYear
    results: list
    explanation: str

def calculate(state):
    return {"results": compare_efls(state["usage"], plans)}

def explain(state):
    results = state["results"]
    difference = results[1].annual_usd - results[0].annual_usd
    if difference == 0:
        text = "The plans tie in this 12-month recurring-cost scenario."
    else:
        text = f"{results[0].plan.name} costs ${difference:,.2f} less over the first 12 months in this scenario."
    text += " Contract length and exit fees are separate tradeoffs; this is not a current-offer recommendation."
    return {"explanation": text}

builder = StateGraph(ComparisonState)
builder.add_node("calculate", calculate)  # compare_efls checks compatibility before arithmetic
builder.add_node("explain", explain)
builder.add_edge(START, "calculate")
builder.add_edge("calculate", "explain")
builder.add_edge("explain", END)
comparison_graph = builder.compile()
usage = load_usage_csv(usage_path)
comparison_state = comparison_graph.invoke({"usage": usage})
results = comparison_state["results"]
print(comparison_state["explanation"])

Reliant Power Savings 2,000 kWh 24 plan costs $147.90 less over the first 12 months in this scenario. Contract length and exit fees are separate tradeoffs; this is not a current-offer recommendation.


## 5. Inspect monthly bills and annual totals
Check how many months actually meet 2,000 kWh. The credit is evaluated month by
month, not against annual average consumption. The breakdown preserves every
component so you can trace an annual total back to each bill.

In [6]:
table(["Plan", "Month", "kWh", "Energy $", "Base $", "Delivery $", "Credit $", "Total $"],
      [(r.plan.name, b.month, b.kwh, b.energy_usd, b.base_usd, b.delivery_usd, b.credit_usd, b.total_usd)
       for r in results for b in r.bills])
table(["Plan", "First 12 months USD", "Credited months", "Contract months", "Stated early exit fee USD"],
      [(r.plan.name, r.annual_usd, sum(b.credit_usd > 0 for b in r.bills), r.plan.contract_months,
        r.plan.termination_usd) for r in results])

Plan,Month,kWh,Energy $,Base $,Delivery $,Credit $,Total $
"Reliant Power Savings 2,000 kWh 24 plan",2025-01,850,131.01,0.00,52.16,0.00,183.17
"Reliant Power Savings 2,000 kWh 24 plan",2025-02,720,110.97,0.00,44.68,0.00,155.65
"Reliant Power Savings 2,000 kWh 24 plan",2025-03,650,100.18,0.00,40.65,0.00,140.83
"Reliant Power Savings 2,000 kWh 24 plan",2025-04,700,107.89,0.00,43.53,0.00,151.42
"Reliant Power Savings 2,000 kWh 24 plan",2025-05,900,138.71,0.00,55.04,0.00,193.75
"Reliant Power Savings 2,000 kWh 24 plan",2025-06,1200,184.95,0.00,72.30,0.00,257.25
"Reliant Power Savings 2,000 kWh 24 plan",2025-07,1500,231.19,0.00,89.57,0.00,320.76
"Reliant Power Savings 2,000 kWh 24 plan",2025-08,1600,246.60,0.00,95.33,0.00,341.93
"Reliant Power Savings 2,000 kWh 24 plan",2025-09,1250,192.66,0.00,75.18,0.00,267.84
"Reliant Power Savings 2,000 kWh 24 plan",2025-10,950,146.42,0.00,57.92,0.00,204.34


Plan,First 12 months USD,Credited months,Contract months,Stated early exit fee USD
"Reliant Power Savings 2,000 kWh 24 plan",2572.69,0,24,295
"Reliant Power Savings 2,000 kWh 12 plan",2720.59,0,12,150


## 6. See exactly when the credit applies
At 1,999 kWh there is no credit. At exactly 2,000 kWh the $150 credit applies.
This is a pricing-rule exercise, not advice to consume more electricity.

In [7]:
table(["Plan", "kWh", "Credit USD", "Bill USD"],
      [(p.name, kwh, b.credit_usd, b.total_usd)
       for p in plans for kwh in [1999, 2000, 2001]
       for b in [calculate_month(MonthlyUsage(month="2025-01", kwh=str(kwh)), p)]])
multiplier = Decimal("1.50")
scenario_usage = UsageYear(months=tuple(MonthlyUsage(month=m.month, kwh=m.kwh * multiplier) for m in usage.months))
scenario_results = comparison_graph.invoke({"usage": scenario_usage})["results"]
table(["Plan", "Annual USD with 50% more usage", "Credited months"],
      [(r.plan.name, r.annual_usd, sum(b.credit_usd > 0 for b in r.bills)) for r in scenario_results])

Plan,kWh,Credit USD,Bill USD
"Reliant Power Savings 2,000 kWh 12 plan",1999,0.00,451.09
"Reliant Power Savings 2,000 kWh 12 plan",2000,150.00,301.32
"Reliant Power Savings 2,000 kWh 12 plan",2001,150.00,301.54
"Reliant Power Savings 2,000 kWh 24 plan",1999,0.00,426.38
"Reliant Power Savings 2,000 kWh 24 plan",2000,150.00,276.60
"Reliant Power Savings 2,000 kWh 24 plan",2001,150.00,276.81


Plan,Annual USD with 50% more usage,Credited months
"Reliant Power Savings 2,000 kWh 24 plan",3539.57,2
"Reliant Power Savings 2,000 kWh 12 plan",3761.48,2


## 7. Understand contract tradeoffs
A lower first-year bill does not make a 24-month commitment identical to a 12-month
one. Listed termination fees apply to early exits with conditions, including a moving
exception. They are **not added to ordinary monthly bills** here. After month 12,
the 12-month plan needs renewal terms we have not modeled. Do not double its
first-year cost to invent a comparable 24-month contract cost.

In [8]:
for plan, record in zip(plans, records):
    print(f"\n{plan.name}: {plan.contract_months} months; stated early termination fee ${plan.termination_usd}")
    print("Model-extracted termination description (check against source):", record.terms.termination_fee.value)


Reliant Power Savings 2,000 kWh 12 plan: 12 months; stated early termination fee $150
Model-extracted termination description (check against source): Yes. $150 early termination fee applies through the end of the contract term. Exception: This fee does not apply if the customer moves and provides a forwarding address and other evidence requested to verify the move.

Reliant Power Savings 2,000 kWh 24 plan: 24 months; stated early termination fee $295
Model-extracted termination description (check against source): $295 early termination fee applies through the end of the contract term; does not apply if the customer moves and provides a forwarding address and other evidence requested to verify the move.


## 8. Save the demonstration report
Save source fingerprints, field evidence, AI extraction issues, assumptions,
synthetic usage and monthly costs. No earlier review or extraction is overwritten.
Rerunning replaces this lesson's report. These candidates remain unapproved.

In [9]:
report = {
    "status": "source_backed_demonstration_not_approved",
    "synthetic_usage": True,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "assumptions": ["Constant EFL snapshot rates for 12 synthetic billing cycles",
        "Standard AEP Central territory; former-Oncor McAllen/Mission exception excluded",
        "Listed recurring delivery totals; no separately estimated ITR adjustments",
        "USD interpreted from Texas context", "Taxes, deposits, late fees and switching fees excluded",
        "Round components to cents half up; credit capped at subtotal"],
    "usage": usage.model_dump(mode="json"),
    "extraction_audit": audit,
    "results": [r.model_dump(mode="json") for r in results],
    "explanation": comparison_state["explanation"],
}
output_dir = PROJECT_ROOT / "output" / "lesson07"
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / "power_savings_comparison.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
assert json.loads(report_path.read_text(encoding="utf-8")) == report
print("Saved:", report_path)

Saved: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\lesson07\power_savings_comparison.json


## Pause and explain
Which plan has the lower energy rate? Why are there no credits in the original
usage scenario? What makes the 24-month term a different commitment?

This notebook uses a narrow, source-checked adapter for these two layouts. Extending
to arbitrary providers needs broader term conversion and review. Notebook 06's
unresolved older Reliant plan remains separate; no missing charges were invented.